In [22]:
import numpy as np
import pandas as pd
from scipy.stats import zscore


In [23]:

base_path = "./Dataset_and_Description/Oxy-files/"
fs = 10.1725   # sampling rate

male_subject_ids = [2, 9, 10, 11, 13, 14, 15, 18, 19, 20]

In [24]:

def create_state_labels(n_samples, fs):
    labels = np.zeros(n_samples, dtype=int)  # 0 = Rest
    
    idx = int(30 * fs)   # start after initial 30 sec rest
    
    for _ in range(10):
        grip_start = idx
        grip_end = idx + int(10 * fs)
        
        grip_start = min(grip_start, n_samples)
        grip_end   = min(grip_end, n_samples)
        
        labels[grip_start:grip_end] = 1  # Grip = 1
        
        idx = grip_end + int(20 * fs)
        if idx >= n_samples:
            break
    
    return labels

In [25]:
from scipy.signal import butter, filtfilt
def bandpass_filter(data, fs, low=0.01, high=0.2, order=4):
    nyq = 0.5 * fs
    b, a = butter(order, [low/nyq, high/nyq], btype='band')
    return filtfilt(b, a, data)

In [26]:
final_dfs = []
only_features=[]

for subject_id in range(1, 16):   # oxy1 → oxy20
    
    # ---------- READ FILE ----------
    file_path = f"{base_path}oxy{subject_id}.csv"
    df = pd.read_csv(file_path)
    
    # ---------- CLEAN ----------
    if "Unnamed: 0" in df.columns:
        df = df.drop(columns=["Unnamed: 0"])

    # df_filtered = df.copy()
    for col in df.columns:
        df[col] = bandpass_filter(df[col].values, fs)

    df = df.rolling(window=5, center=True).median()
    df = df.dropna()
    
    # ---------- SELECT CHANNELS SAFELY ----------
    # channel_cols = [col for col in df.columns if col.startswith("ch")]
    # channel_data = df[channel_cols]
    
    # ---------- ✅ Z-SCORE NORMALIZATION (PER CHANNEL, PER SUBJECT) ----------
    df = df.apply(zscore)
    
    # ---------- ✅ CREATE STATE ----------
    df["State"] = create_state_labels(len(df), fs)
    
    # ---------- ✅ ROW-WISE FEATURES (ON NORMALIZED DATA) ----------
    df["Mean_Row"]   = df.mean(axis=1)
    df["Std_Row"]    = df.std(axis=1)
    df["Energy_Row"] = (df** 2).sum(axis=1)
    df["RMS_Row"]    = np.sqrt((df ** 2).mean(axis=1))
    
    # ---------- ✅ ADD GENDER ----------
    if subject_id in male_subject_ids:
        df["Gender"] = 1   # Male
    else:
        df["Gender"] = 0   # Female
    
    # ---------- ✅ KEEP ONLY FINAL ML COLUMNS ----------
    final_df = df[[
        "Mean_Row",
        "Std_Row",
        "Energy_Row",
        "RMS_Row",
        "State",
        "Gender"
    ]]
    
    final_dfs.append(df)
    only_features.append(final_df)

In [27]:
final_dfs_test = []
only_features_test=[]

for subject_id in range(16, 21):   # oxy1 → oxy20
    
    # ---------- READ FILE ----------
    file_path = f"{base_path}oxy{subject_id}.csv"
    df = pd.read_csv(file_path)
    
    # ---------- CLEAN ----------
    if "Unnamed: 0" in df.columns:
        df = df.drop(columns=["Unnamed: 0"])

    # df_filtered = df.copy()
    for col in df.columns:
        df[col] = bandpass_filter(df[col].values, fs)

    df = df.rolling(window=5, center=True).median()
    df = df.dropna()
    
    # ---------- SELECT CHANNELS SAFELY ----------
    # channel_cols = [col for col in df.columns if col.startswith("ch")]
    # channel_data = df[channel_cols]
    
    # ---------- ✅ Z-SCORE NORMALIZATION (PER CHANNEL, PER SUBJECT) ----------
    df = df.apply(zscore)
    
    # ---------- ✅ CREATE STATE ----------
    df["State"] = create_state_labels(len(df), fs)
    
    # ---------- ✅ ROW-WISE FEATURES (ON NORMALIZED DATA) ----------
    df["Mean_Row"]   = df.mean(axis=1)
    df["Std_Row"]    = df.std(axis=1)
    df["Energy_Row"] = (df** 2).sum(axis=1)
    df["RMS_Row"]    = np.sqrt((df ** 2).mean(axis=1))
    
    # ---------- ✅ ADD GENDER ----------
    if subject_id in male_subject_ids:
        df["Gender"] = 1   # Male
    else:
        df["Gender"] = 0   # Female
    
    # ---------- ✅ KEEP ONLY FINAL ML COLUMNS ----------
    final_df = df[[
        "Mean_Row",
        "Std_Row",
        "Energy_Row",
        "RMS_Row",
        "State",
        "Gender"
    ]]
    
    final_dfs_test.append(df)
    only_features_test.append(final_df)

In [30]:
final_dataset = pd.concat(final_dfs, ignore_index=True)
final_dataset_test = pd.concat(final_dfs_test, ignore_index=True)


In [31]:
print("✅ FINAL DATASET SHAPE:", final_dataset.shape)

✅ FINAL DATASET SHAPE: (51660, 26)


In [32]:
final_dataset.head()

,ch1,ch2,ch3,ch4,ch5,ch6,ch7,ch8,ch9,ch10,...,ch17,ch18,ch19,ch20,State,Mean_Row,Std_Row,Energy_Row,RMS_Row,Gender
0,0.066503,-0.100691,0.049500,0.675047,0.243700,0.911247,0.202707,-0.165607,0.173002,0.505412,...,0.290500,0.640507,0.263046,0.048925,0,0.240190,0.274402,2.925736,0.691787,0
1,0.085629,-0.072208,0.079822,0.718774,0.263597,0.987105,0.250586,-0.172465,0.208833,0.564219,...,0.338784,0.685072,0.305551,0.063335,0,0.272498,0.291644,3.504852,0.811091,0
2,0.104044,-0.044105,0.110033,0.762034,0.283589,1.062907,0.298415,-0.179036,0.244501,0.622519,...,0.386852,0.729381,0.347959,0.077538,0,0.304774,0.309400,4.149548,0.943581,0
3,0.121609,-0.016520,0.140061,0.804719,0.303732,1.138529,0.346135,-0.185191,0.279916,0.680073,...,0.434553,0.773274,0.390138,0.091438,0,0.336975,0.327470,4.857352,1.088792,0
4,0.138192,0.010422,0.169834,0.846720,0.324082,1.213846,0.393685,-0.190804,0.314996,0.736651,...,0.481745,0.816599,0.431955,0.104941,0,0.369055,0.345673,5.625219,1.246133,0


In [33]:
X = final_dataset.drop(columns=["State"])
y = final_dataset["State"]
gender = final_dataset["Gender"]



In [34]:
X_test = final_dataset_test.drop(columns=["State"])
y_test = final_dataset_test["State"]
gender_test = final_dataset_test["Gender"]

In [41]:
from sklearn.model_selection import train_test_split

# ✅ 1) First split: 60% Train | 40% Temp  (stratified by Gender)
X_train, X_val, y_train, y_val, gender_train, gender_val = train_test_split(
    X, y, gender,
    test_size=0.2,
    random_state=42,
    stratify=gender
)

# ✅ 2) Second split: 20% Val | 20% Test (also stratified by Gender)
# X_val, X_test, y_val, y_test, gender_val, gender_test = train_test_split(
#     X_temp, y_temp, gender_temp,
#     test_size=0.5,   # half of 40% → 20% & 20%
#     random_state=42,
#     stratify=gender_temp
# )


In [42]:
print("TRAIN Gender Ratio:\n", gender_train.value_counts(normalize=True))
print("\nVAL Gender Ratio:\n", gender_val.value_counts(normalize=True))
print("\nTEST Gender Ratio:\n", gender_test.value_counts(normalize=True))

TRAIN Gender Ratio:
 Gender
0    0.533343
1    0.466657
Name: proportion, dtype: float64

VAL Gender Ratio:
 Gender
0    0.533295
1    0.466705
Name: proportion, dtype: float64

TEST Gender Ratio:
 Gender
1    0.6
0    0.4
Name: proportion, dtype: float64


In [44]:
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(
    n_neighbors=3,     # K = 5 (good default)
    weights="distance",
    metric= 'manhattan'  # closer neighbors matter more
)

knn.fit(X_train, y_train)


,n_neighbors,3
,weights,'distance'
,algorithm,'auto'
,leaf_size,30
,p,2
,metric,'manhattan'
,metric_params,None
,n_jobs,None


In [45]:
print("✅ KNN Train Accuracy:", knn.score(X_train, y_train))
print("✅ KNN Val Accuracy:  ", knn.score(X_val, y_val))
print("✅ KNN Test Accuracy: ", knn.score(X_test, y_test))


✅ KNN Train Accuracy: 1.0
✅ KNN Val Accuracy:   0.9875145180023229
✅ KNN Test Accuracy:  0.7875145180023229


In [19]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "n_neighbors": [3, 5, 7, 9, 11],
    "weights": ["uniform", "distance"],
    "metric": ["euclidean", "manhattan"]
}

knn = KNeighborsClassifier()

grid_knn = GridSearchCV(
    knn,
    param_grid,
    cv=3,
    scoring="accuracy",
    n_jobs=-1,
    verbose=2
)

grid_knn.fit(X_train, y_train)

print("✅ Best KNN Params:", grid_knn.best_params_)
print("✅ Best CV Accuracy:", grid_knn.best_score_)


Fitting 3 folds for each of 20 candidates, totalling 60 fits
[CV] END ..metric=euclidean, n_neighbors=3, weights=distance; total time=   6.7s
[CV] END ..metric=euclidean, n_neighbors=3, weights=distance; total time=   7.1s
[CV] END ..metric=euclidean, n_neighbors=3, weights=distance; total time=   7.3s
[CV] END ...metric=euclidean, n_neighbors=3, weights=uniform; total time=   8.3s
[CV] END ...metric=euclidean, n_neighbors=5, weights=uniform; total time=   8.5s
[CV] END ...metric=euclidean, n_neighbors=3, weights=uniform; total time=   8.7s
[CV] END ...metric=euclidean, n_neighbors=3, weights=uniform; total time=   9.2s
[CV] END ...metric=euclidean, n_neighbors=5, weights=uniform; total time=   9.8s
[CV] END ..metric=euclidean, n_neighbors=5, weights=distance; total time=   6.7s
[CV] END ..metric=euclidean, n_neighbors=5, weights=distance; total time=   7.0s
[CV] END ..metric=euclidean, n_neighbors=5, weights=distance; total time=   7.5s
[CV] END ...metric=euclidean, n_neighbors=5, wei

In [ ]:
best_knn = grid_knn.best_estimator_

print("✅ Final KNN Train:", best_knn.score(X_train, y_train))
# print("✅ Final KNN Val:  ", best_knn.score(X_val, y_val))
print("✅ Final KNN Test: ", best_knn.score(X_test, y_test))


✅ Final KNN Train: 1.0
✅ Final KNN Val:   0.9796022067363531
✅ Final KNN Test:  0.9809814169570267


In [21]:
import pickle

with open("knn_model.pkl", "wb") as f:
    pickle.dump(best_knn, f)

print("✅ KNN model saved as knn_model.pkl")


✅ KNN model saved as knn_model.pkl
